In [ ]:
import glob
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# 1. Load Renewables.ninja PV data (2014-2018)
ninja_files = sorted(glob.glob("data/ninja_pv_*.csv"))
df_ninja = pd.concat(
    [pd.read_csv(f, skiprows=3) for f in ninja_files], ignore_index=True
)
df_ninja["time"] = pd.to_datetime(df_ninja["time"]).dt.tz_localize(None)


# 2. Parse SMHI files
def load_smhi_file(filepath, val_col_name):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        skip_header = next(
            (
                i
                for i, l in enumerate(f)
                if l.startswith(("Datum", "Date"))
            ),
            0,
        )

    df = pd.read_csv(
        filepath,
        skiprows=skip_header,
        sep=";",
        encoding="utf-8",
        on_bad_lines="skip",
    )
    df.columns = [c.strip() for c in df.columns]
    time_cols = (
        ("Datum", "Tid (UTC)") if "Datum" in df.columns else ("Date", "Time")
    )
    df["time"] = pd.to_datetime(df[time_cols[0]] + " " + df[time_cols[1]])

    val_cols = [
        c
        for c in df.columns
        if c not in ["Datum", "Tid (UTC)", "Date", "Time", "Kvalitet", "Quality"]
    ]
    target_col = val_cols[0] if val_cols else df.columns[2]

    df_clean = df[["time", target_col]].rename(
        columns={target_col: val_col_name}
    )
    df_clean[val_col_name] = pd.to_numeric(
        df_clean[val_col_name], errors="coerce"
    )
    return df_clean


# 3. Merge SMHI features with PV data
smhi_files = {
    "temperature": "data/smhi-opendata_air_temperture_201401_201812.csv",
    "cloud_cover": "data/smhi-opendata_cloud_amount_201401_201812.csv",
    "wind_speed": "data/smhi-opendata_wind_speed_201401_201812.csv",
    "humidity": "data/smhi-opendata_relative_humidity_201401_201812.csv",
    "pressure": "data/smhi-opendata_air_pressure_201401_201812.csv",
}

df_final = df_ninja.drop(columns=["temperature"], errors="ignore")
for col, path in smhi_files.items():
    df_final = df_final.merge(load_smhi_file(path, col), on="time", how="left")

# 4. Clean missing values and add cyclical time + lag features
df_final = (
    df_final.sort_values("time")
    .interpolate(method="linear")
    .bfill()
    .ffill()
    .reset_index(drop=True)
)
df_final["electricity"] = df_final["electricity"].clip(lower=0.0)

df_final["hour_sin"] = np.sin(2 * np.pi * df_final["time"].dt.hour / 24.0)
df_final["hour_cos"] = np.cos(2 * np.pi * df_final["time"].dt.hour / 24.0)
df_final["doy_sin"] = np.sin(2 * np.pi * df_final["time"].dt.dayofyear / 365.25)
df_final["doy_cos"] = np.cos(2 * np.pi * df_final["time"].dt.dayofyear / 365.25)
df_final["pv_lag_1h"] = df_final["electricity"].shift(1)
df_final["pv_lag_24h"] = df_final["electricity"].shift(24)
df_final = df_final.dropna().reset_index(drop=True)

print(f"Dataset ready! Total rows: {len(df_final)}")
df_final.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# 1. Select features & targets
feature_cols = [
    "temperature",
    "cloud_cover",
    "wind_speed",
    "humidity",
    "pressure",
    "hour_sin",
    "hour_cos",
    "doy_sin",
    "doy_cos",
    "pv_lag_1h",
    "pv_lag_24h",
]

X = df_final[feature_cols].values
y = df_final["electricity"].values
y_binary = (y > 0.001).astype(int)

# 2. Chronological 80/20 Train-Test Split & Leakage-Free Scaling
split = int(0.80 * len(df_final))
X_train_raw, X_test_raw = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
y_train_b, y_test_b = y_binary[:split], y_binary[split:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

tscv = TimeSeriesSplit(n_splits=5)

# 3. Hyperparameter Tuning & Training (LR, SNN, DNN)
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {"C": [0.1, 1.0, 10.0]},
    cv=tscv,
    scoring="f1",
    n_jobs=-1,
).fit(X_train, y_train_b)

snn_grid = GridSearchCV(
    MLPRegressor(max_iter=300, early_stopping=True, random_state=42),
    {"hidden_layer_sizes": [(32,), (64,)], "alpha": [0.0001, 0.001]},
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
).fit(X_train, y_train)

dnn_grid = GridSearchCV(
    MLPRegressor(max_iter=300, early_stopping=True, random_state=42),
    {"hidden_layer_sizes": [(64, 32), (128, 64, 32)], "alpha": [0.0001, 0.001]},
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
).fit(X_train, y_train)

# 4. Evaluate on Unseen Test Set
y_pred_lr = lr_grid.predict(X_test)
y_pred_snn = np.clip(snn_grid.predict(X_test), 0, None)
y_pred_dnn = np.clip(dnn_grid.predict(X_test), 0, None)

results_summary = {
    f"Logistic Regression (C={lr_grid.best_params_['C']})": {
        "MAE": "-",
        "RMSE": "-",
        "R2 Score": "-",
        "Accuracy": f"{accuracy_score(y_test_b, y_pred_lr):.4f}",
        "F1-Score": f"{f1_score(y_test_b, y_pred_lr):.4f}",
    },
    f"Shallow NN {snn_grid.best_params_['hidden_layer_sizes']}": {
        "MAE": f"{mean_absolute_error(y_test, y_pred_snn):.4f}",
        "RMSE": f"{np.sqrt(mean_squared_error(y_test, y_pred_snn)):.4f}",
        "R2 Score": f"{r2_score(y_test, y_pred_snn):.4f}",
        "Accuracy": f"{accuracy_score(y_test_b, (y_pred_snn > 0.001).astype(int)):.4f}",
        "F1-Score": f"{f1_score(y_test_b, (y_pred_snn > 0.001).astype(int)):.4f}",
    },
    f"Deep NN {dnn_grid.best_params_['hidden_layer_sizes']}": {
        "MAE": f"{mean_absolute_error(y_test, y_pred_dnn):.4f}",
        "RMSE": f"{np.sqrt(mean_squared_error(y_test, y_pred_dnn)):.4f}",
        "R2 Score": f"{r2_score(y_test, y_pred_dnn):.4f}",
        "Accuracy": f"{accuracy_score(y_test_b, (y_pred_dnn > 0.001).astype(int)):.4f}",
        "F1-Score": f"{f1_score(y_test_b, (y_pred_dnn > 0.001).astype(int)):.4f}",
    },
}

df_results = pd.DataFrame.from_dict(results_summary, orient="index")
print("=== Final Model Performance Comparison ===")
print(df_results)

In [ ]:
import matplotlib.pyplot as plt

# Select a 7-day window (168 hours) in the test set where solar generation is active
test_times = df_final["time"].iloc[split:].values
start_idx = 2500  # Adjust index to view different weeks (e.g., summer vs. winter)
window = slice(start_idx, start_idx + 168)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [2, 1]})

# 1. Time-Series Plot (7-Day Window)
ax1.plot(test_times[window], y_test[window], label="Actual PV Power", color="black", linewidth=2)
ax1.plot(test_times[window], y_pred_snn[window], label="SNN (1 Layer)", color="#1954A6", linestyle="--", alpha=0.85)
ax1.plot(test_times[window], y_pred_dnn[window], label="DNN (Multi-Layer)", color="#E69F00", linewidth=1.5)
ax1.set_title("Gothenburg Solar Power Forecast (7-Day Test Window)")
ax1.set_xlabel("Date")
ax1.set_ylabel("Normalized PV Generation (kW)")
ax1.legend()
ax1.grid(True, alpha=0.3)
fig.autofmt_xdate()

# 2. Parity Scatter Plot (Actual vs. DNN Predicted)
ax2.scatter(y_test, y_pred_dnn, alpha=0.15, color="#002F5F", s=10, label="Test Hours")
ax2.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1.5, label="Ideal Fit (y = x)")
ax2.set_title("Actual vs. DNN Predicted Generation")
ax2.set_xlabel("Actual PV Power (kW)")
ax2.set_ylabel("Predicted PV Power (kW)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("solar_forecast_comparison.png", dpi=300)
plt.show()

In [ ]:
# Evaluate Seasonal & Directional Bias on the Test Set (Step 7 Analysis)
df_bias = pd.DataFrame({
    "month": df_final["time"].iloc[split:].dt.month.values,
    "actual": y_test,
    "pred_dnn": y_pred_dnn,
})
df_bias["error"] = (
    df_bias["pred_dnn"] - df_bias["actual"]
)  # Positive = Overprediction

# Compare Summer (May-Aug) vs. Winter (Nov-Feb) accuracy
summer = df_bias[df_bias["month"].isin([5, 6, 7, 8])]
winter = df_bias[df_bias["month"].isin([11, 12, 1, 2])]

overpred_rate = (df_bias["error"] > 0.01).mean() * 100

print("=== Step 7: Empirical Bias & Seasonal Inaccuracy Analysis (DNN) ===")
print(
    f"Summer (May-Aug) | MAE: {mean_absolute_error(summer['actual'], summer['pred_dnn']):.4f} kW | R2: {r2_score(summer['actual'], summer['pred_dnn']):.4f}"
)
print(
    f"Winter (Nov-Feb) | MAE: {mean_absolute_error(winter['actual'], winter['pred_dnn']):.4f} kW | R2: {r2_score(winter['actual'], winter['pred_dnn']):.4f}"
)
print(
    f"Mean Bias Error (MBE): {df_bias['error'].mean():+.4f} kW (Overprediction frequency: {overpred_rate:.1f}% of hours)"
)

df_final.corr(numeric_only=True)['electricity'].sort_values(ascending=False)